In [1]:
!pip install mlflow dagshub torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 64.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 94.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import mlflow
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

In [3]:
# Настройка подключения к DagsHub
os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/ladaogneva13/my-first-repo.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = 'ladaogneva13'  # ваш username на DagsHub
os.environ['MLFLOW_TRACKING_PASSWORD'] = '49791b957fb5abd18897a9a4e017280d3b205521'  # ваш токен из DagsHub

In [4]:
# Пример архитектуры
class Net(nn.Module):
    def __init__(self, hidden_size, dropout, activation):
        super().__init__()
        self.fc1 = nn.Linear(32*32*3, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.act = getattr(torch.nn, activation)()
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

def train(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for data, targets in loader:
        data, targets = data.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * data.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for data, targets in loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * data.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
    return running_loss / total, correct / total

В случае нелокального запуска для mlflow нужно залогиниться на специальном сервере, где будут храниться метрики и артефакты обучения.

Платформа Databricks предоставляет бесплатный тариф для создания удаленного mlflow-сервера. Сайт [Databricks](https://www.databricks.com/) доступен только через VPN. Создайте учетную запись на этой платформе, либо запускайте код локально с открытием mlflow-сервера на localhost.

Еще есть альтернатива [dagshub](https://dagshub.com/), где тоже можно бесплатно создать учетную запись под удаленный mlflow-сервер. 

In [5]:
# Инициализация MLflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment("CIFAR10_Classification")

2025/06/07 06:23:57 INFO mlflow.tracking.fluent: Experiment with name 'CIFAR10_Classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/ffd6b171b3984e2eb2556aa4ad8c7d80', creation_time=1749277437719, experiment_id='0', last_update_time=1749277437719, lifecycle_stage='active', name='CIFAR10_Classification', tags={}>

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = {
    'epochs': 10, 'lr': 0.001, 'batch_size': 64,
    'hidden_size': 128, 'dropout': 0.5, 'activation': 'ReLU'
}
transform = transforms.ToTensor()
train_set = datasets.CIFAR10(root='.', train=True, download=True, transform=transform)
val_set = datasets.CIFAR10(root='.', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=config['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=config['batch_size'])

with mlflow.start_run():
    mlflow.log_params(config)
    model = Net(config['hidden_size'], config['dropout'], config['activation']).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.CrossEntropyLoss()
    for epoch in range(config['epochs']):
        loss, acc = train(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        mlflow.log_metrics({'train_loss': loss, 'train_acc': acc, 'val_loss': val_loss, 'val_acc': val_acc}, step=epoch)
        print(f'Epoch {epoch+1}: train loss={loss:.4f}, train acc={acc:.4f}; val loss={val_loss:.4f}, val acc={val_acc:.4f}')
    
    # Логирование модели
    mlflow.pytorch.log_model(model, "model")

100%|██████████| 170M/170M [00:04<00:00, 34.8MB/s] 


Epoch 1: train loss=2.1211, train acc=0.1915; val loss=1.9850, val acc=0.2660
Epoch 2: train loss=2.0613, train acc=0.2093; val loss=1.9255, val acc=0.2843
Epoch 3: train loss=2.0438, train acc=0.2222; val loss=1.9406, val acc=0.2892
Epoch 4: train loss=2.0375, train acc=0.2261; val loss=1.9230, val acc=0.3056
Epoch 5: train loss=2.0340, train acc=0.2291; val loss=1.9050, val acc=0.3127
Epoch 6: train loss=2.0268, train acc=0.2346; val loss=1.8996, val acc=0.2966
Epoch 7: train loss=2.0225, train acc=0.2353; val loss=1.9307, val acc=0.3142
Epoch 8: train loss=2.0153, train acc=0.2372; val loss=1.8620, val acc=0.3352
Epoch 9: train loss=2.0180, train acc=0.2386; val loss=1.8807, val acc=0.3112
Epoch 10: train loss=2.0139, train acc=0.2391; val loss=1.9321, val acc=0.3161


2025/06/07 06:25:39 WARNING mlflow.utils.requirements_utils: Found torch version (2.6.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torch==2.6.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/06/07 06:25:53 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu124) contains a local version label (+cu124). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2025/06/07 06:25:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run upbeat-elk-343 at: https://dagshub.com/ladaogneva13/my-first-repo.mlflow/#/experiments/0/runs/b4840de60ee94d458f84565edb5432f4
🧪 View experiment at: https://dagshub.com/ladaogneva13/my-first-repo.mlflow/#/experiments/0
